In [1]:
%matplotlib inline
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

from xrd_data import MATERIALS, PALETTE, WAVELENGTHS
from xrd_pemwe import make_figure, save_pdf, save_gif, NATURE_RC

plt.rcParams.update(NATURE_RC)
OUT_DIR = Path('.')

material_names = list(MATERIALS.keys())
checks = [W.Checkbox(value=True, description=n, indent=False,
                     layout=W.Layout(width='180px'))
          for n in material_names]

wavelength = W.Dropdown(options=list(WAVELENGTHS.keys()), value='Cu Kα',
                       description='λ', layout=W.Layout(width='220px'))
crystallite = W.FloatSlider(value=30.0, min=2.0, max=200.0, step=1.0,
                            description='D (nm)', readout_format='.0f',
                            continuous_update=False,
                            layout=W.Layout(width='340px'))
tt_range = W.FloatRangeSlider(value=[20, 90], min=10, max=120, step=1,
                              description='2θ', continuous_update=False,
                              layout=W.Layout(width='340px'))
eta = W.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                    description='η (PV)', continuous_update=False,
                    layout=W.Layout(width='340px'))
layout_mode = W.ToggleButtons(options=[('stacked', 'stack'), ('overlay', 'overlay')],
                              value='stack', description='layout',
                              layout=W.Layout(width='280px'))
show_profile = W.Checkbox(value=True, description='profile', indent=False)
show_sticks = W.Checkbox(value=True, description='sticks (hkl)', indent=False)

save_pdf_btn = W.Button(description='save PDF', layout=W.Layout(width='130px'))
save_gif_btn = W.Button(description='save GIF', layout=W.Layout(width='130px'))
status = W.HTML(value='')

fig_out = W.Output()


def selected_materials():
    return [n for c, n in zip(checks, material_names) if c.value]


def redraw(*_):
    sel = selected_materials()
    with fig_out:
        clear_output(wait=True)
        if not sel:
            fig, ax = plt.subplots(figsize=(3.6, 2.2))
            ax.text(0.5, 0.5, 'select at least one phase', ha='center', va='center',
                    transform=ax.transAxes, color='#555')
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
            plt.show()
            plt.close(fig)
            return
        fig, _ = make_figure(
            selected=sel,
            wavelength_A=WAVELENGTHS[wavelength.value],
            crystallite_nm=crystallite.value,
            eta=eta.value,
            tt_range=tuple(tt_range.value),
            show_profile=show_profile.value,
            show_sticks=show_sticks.value,
            stack=(layout_mode.value == 'stack'),
        )
        plt.show()
        plt.close(fig)


for w in checks + [wavelength, crystallite, tt_range, eta, layout_mode, show_profile, show_sticks]:
    w.observe(redraw, names='value')


def on_save_pdf(_):
    sel = selected_materials()
    if not sel:
        status.value = "<span style='color:#b00'>nothing selected</span>"
        return
    path = OUT_DIR / 'xrd_pemwe.pdf'
    save_pdf(path, selected=sel,
             wavelength_A=WAVELENGTHS[wavelength.value],
             crystallite_nm=crystallite.value, eta=eta.value,
             tt_range=tuple(tt_range.value),
             show_profile=show_profile.value,
             show_sticks=show_sticks.value,
             stack=(layout_mode.value == 'stack'))
    status.value = f"<span style='color:#060'>wrote {path}</span>"


def on_save_gif(_):
    sel = selected_materials()
    if not sel:
        status.value = "<span style='color:#b00'>nothing selected</span>"
        return
    status.value = '<span>rendering GIF…</span>'
    path = OUT_DIR / 'xrd_pemwe.gif'
    save_gif(path, selected=sel,
             wavelength_A=WAVELENGTHS[wavelength.value],
             crystallite_nm=crystallite.value, eta=eta.value,
             tt_range=tuple(tt_range.value),
             show_sticks=show_sticks.value)
    status.value = f"<span style='color:#060'>wrote {path}</span>"


save_pdf_btn.on_click(on_save_pdf)
save_gif_btn.on_click(on_save_gif)

controls = W.VBox([
    W.HBox([W.Label('phases:'), W.VBox(checks)]),
    W.HBox([wavelength, layout_mode]),
    crystallite,
    tt_range,
    eta,
    W.HBox([show_profile, show_sticks]),
    W.HBox([save_pdf_btn, save_gif_btn, status]),
])

display(W.HBox([controls, fig_out]))
redraw()